# POLARIS HEALPix → IQUV FITS
Minimal conversion for the existing **near2 dust** output: one wavelength, native RING indexing, flux in Jy/pixel.

The output is an explicit sky table (`PIXEL`, `L_DEG`, `B_DEG`, `I`, `Q`, `U`, `V`) in MJy/sr, not a rectangular image. Native angular sampling is retained; no interpolation, smoothing, background subtraction, or checks are performed.

This dataset has zero I outside the computed footprint; those rows are omitted. Its native convention requires `l = 180° − phi` and both Q and U to change sign for the IAU basis used by the existing products. These conversions are specific to this workflow, not a universal POLARIS rule.

Requirements: NumPy, Astropy, astropy-healpix. All code is included below.

V retains its native POLARIS sign; only Q and U are converted to the existing IAU linear-polarization basis.


## 1 · Files
Change these two paths. The output file is overwritten if it already exists.


In [ ]:
from pathlib import Path
ROOT = Path('/home/jcwho/work/work/HI_fila/HI_analasys_2026/soler2022synthetic/depth_test_R4p5_short_medium')
INPUT = ROOT/'runs/near2_dust_production/data/polaris_detector_nr0001.fits.gz'
OUTPUT = Path('/home/jcwho/work/work/HI_fila/polarisTutor/near2_IQUV_minimal.fits')


## 2 · Read and convert
Divide Jy/pixel by the pixel solid angle and by 10⁶ to obtain MJy/sr. Retain the total I, including any background already present in the raw output.


In [ ]:
import numpy as np
from astropy.io import fits
from astropy_healpix import HEALPix

with fits.open(INPUT) as hdul:
    nside = hdul[1].header['NSIDE']
    data = hdul[1].data
    pixel = np.flatnonzero(data.field(0) != 0)
    iquv = np.array([data.field(j)[pixel] for j in range(4)])



omega = 4*np.pi/(12*nside**2)
iquv /= omega*1e6
iquv[1:3] *= -1
phi, lat = HEALPix(nside=nside, order='ring').healpix_to_lonlat(pixel)
lon = (180-phi.deg+180)%360-180


In [ ]:
nside
#print(iquv.shape)

## 3 · Write IQUV FITS
`PIXEL` remains a native observer-frame index, not a standard Galactic HEALPix index. Use `L_DEG` and `B_DEG` for this workflow's sky coordinates. The Galactic-coordinate IAU polarization basis follows the existing near2 conversion.


In [ ]:
columns = [fits.Column(name='PIXEL', format='K', array=pixel),
           fits.Column(name='L_DEG', format='D', unit='deg', array=lon),
           fits.Column(name='B_DEG', format='D', unit='deg', array=lat.deg)]
columns += [fits.Column(name=name, format='D', unit='MJy/sr', array=values)
            for name, values in zip(['I','Q','U','V'], iquv)]
sky = fits.BinTableHDU.from_columns(columns, name='SKY')
sky.header.update(NSIDE=nside, ORDERING='RING', INDXSCHM='EXPLICIT',
                  PIXTYPE='HEALPIX', PIXFRAME='NATIVE', POLCCONV='IAU')

fits.HDUList([fits.PrimaryHDU(), sky]).writeto(OUTPUT, overwrite=True)


## 4 · Display I, Q, U, V
Show all retained directions without interpolation. I uses a power stretch; Q and U share a symmetric logarithmic scale. V has its own symmetric scale to reveal weaker circular polarization and retains the native sign. A numerical scale floor allows an all-zero V map to be displayed. Colour stretches do not change the FITS values. Longitude increases to the left.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import PowerNorm, SymLogNorm

plt.rcParams.update({'font.family':'sans-serif', 'font.sans-serif':['DejaVu Sans'],
                     'font.size':18, 'axes.labelsize':20, 'axes.titlesize':22})
fig, axes = plt.subplots(2, 2, figsize=(13, 10), layout='constrained', sharex=True, sharey=True)
qu_limit = max(abs(iquv[1:3]).max(), np.finfo(float).eps)
v_limit = max(abs(iquv[3]).max(), np.finfo(float).eps)
for j, (ax, name) in enumerate(zip(axes.flat, ['I', 'Q', 'U', 'V'])):
    limit = v_limit if j == 3 else qu_limit
    norm = PowerNorm(0.4) if j == 0 else SymLogNorm(limit*0.02, vmin=-limit, vmax=limit)
    image = ax.scatter(lon, lat.deg, c=iquv[j], s=2, linewidths=0,
                       cmap='inferno' if j == 0 else 'RdBu_r', norm=norm)
    ax.set(title=name, aspect='equal')
    fig.colorbar(image, ax=ax, shrink=0.8, label='MJy/sr')
for ax in axes[1]:
    ax.set_xlabel(r'$l$ (deg)')
for ax in axes[:,0]:
    ax.set_ylabel(r'$b$ (deg)')
axes[0,0].invert_xaxis()
fig.savefig(Path(OUTPUT).with_name('near2_IQUV.png'), dpi=300)
plt.show()
